In [1]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00:00:0100:01


In [2]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
    IN_COLAB = True
except:
    IN_COLAB = False
    print("Not running in Colab")

Mounted at /content/drive


In [3]:
# Import libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from typing import Dict, Any, List
import pandas as pd
import ast
import re

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

model_path = "/content/drive/MyDrive/hf_models/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Model loaded successfully")
print(f"✅ Device: {device}")
print(f"✅ Tokenizer loaded: {tokenizer is not None}")
print(f"✅ Model loaded: {model is not None}")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ Model loaded locally from Drive
✅ Model loaded successfully
✅ Device: cuda
✅ Tokenizer loaded: True
✅ Model loaded: True


In [5]:
def ask_llm(messages, max_new_tokens=256, do_sample=False):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [6]:
state_template = {
    "request_received": False,
    "request_classified": False,
    "authorized": None,
    "analysis_done": False,
    "result": None,
    "answered": False,
    "rejection_reason": None
}

In [7]:
DENY_KEYWORDS = [
    "show", "list", "export", "download", "all rows",
    "all records", "entire dataset"
]

def check_authorization(question: str) -> (bool, str):
    q_lower = question.lower()
    for kw in DENY_KEYWORDS:
        if kw in q_lower:
            return False, f"Contains forbidden keyword '{kw}'"
    return True, ""

In [8]:
def build_code_prompt(question, df):
    schema = {
        "columns": list(df.columns),
        "dtypes": {col: str(df[col].dtype) for col in df.columns}
    }

    return [
        {
        "role": "system",
        "content": f"""
        You are a data analyst working with a pandas DataFrame named df.

        Dataset schema:
        {schema}

        STRICT RULES:
        - Use ONLY existing column names exactly as written.
        - Do NOT import anything.
        - Do NOT define functions.
        - Do NOT use loops.
        - Use pandas only.
        - Store final answer in variable named result.
        - Output ONLY executable python code.
        - Do NOT include explanations, comments, or markdown.
        """
        },
        {
        "role": "user",
        "content": question
        }
    ]

In [9]:
def extract_code(text):

    text = text.replace("```python", "").replace("```", "")
    cleaned_lines = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("#") or line == "":
            continue
        if line.startswith("print"):
            continue
        line = re.sub(r"\.sort_values\(by=", ".sort_values(", line)
        if re.search(r"(^result\s*=)|(^df\.)|(^pd\.)|(^result\.)", line):
            cleaned_lines.append(line)
    if not any("result" in l for l in cleaned_lines):
        raise ValueError("No valid 'result =' line found.")
    return "\n".join(cleaned_lines)

In [10]:
FORBIDDEN_NODES = (
    ast.Import, ast.ImportFrom, ast.With,
    ast.While, ast.For, ast.Try,
    ast.FunctionDef, ast.ClassDef, ast.Delete
)

FORBIDDEN_NAMES = {
    "exec", "eval", "open", "__import__", "compile",
    "os", "sys", "subprocess", "shutil", "print"
}

def validate_code_safety(code: str):
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, FORBIDDEN_NODES):
            raise ValueError(f"Forbidden operation: {type(node).__name__}")
        if isinstance(node, ast.Name) and node.id in FORBIDDEN_NAMES:
            raise ValueError(f"Forbidden name used: {node.id}")

In [11]:
def run_generated_code(code, df):
    validate_code_safety(code)
    safe_globals = {"__builtins__": {}, "df": df, "pd": pd}
    safe_locals = {}
    exec(code, safe_globals, safe_locals)
    if "result" not in safe_locals:
        raise ValueError("Code must assign result variable.")
    return safe_locals["result"]

In [12]:
def explain_result(question, result):

    # تحويل pandas Series إلى dict
    if isinstance(result, pd.Series):
        result = result.to_dict()

    # تحويل DataFrame صغير إلى dict
    elif isinstance(result, pd.DataFrame):
        if len(result) <= 50:
            result = result.to_dict(orient="records")
        else:
            result = f"DataFrame with shape {result.shape}"

    return f"Answer to '{question}': {result}"

In [13]:
ALLOWED_ACTIONS = ["classify_request", "run_analysis", "reject_request", "answer_user", "finish"]

def decide_action(state: dict, question: str) -> str:
    if not state["request_classified"]:
        return "classify_request"
    if state["authorized"] and not state["analysis_done"]:
        return "run_analysis"
    if state["analysis_done"] and not state["answered"]:
        return "answer_user"
    if state["answered"]:
        return "finish"
    return "reject_request"

In [14]:
def execute_action(action: str, state: dict, df: pd.DataFrame, question: str):

    print(f"\n[Step] Current action: {action}")
    if action == "classify_request":
        auth, reason = check_authorization(question)
        state["request_classified"] = True
        state["authorized"] = auth
        if not auth:
            state["rejection_reason"] = reason
        print(f"[Step] Classification done – authorized={state['authorized']}")

    elif action == "run_analysis":
        if state["authorized"] is False:
            action = "reject_request"
        else:
            print("[Step] Asking LLM to generate pandas code...")
            prompt = build_code_prompt(question, df)
            raw_output = ask_llm(prompt)
            print("\n[LLM Raw Output]")
            print(raw_output)
            code = extract_code(raw_output)
            print("\n[Cleaned Code]")
            print(code)
            result = run_generated_code(code, df)
            state["result"] = result
            state["analysis_done"] = True
            print("[Step] Analysis done – result ready")

    elif action == "answer_user":
        explanation = explain_result(question, state["result"])
        print("\nAnswer:", explanation)
        state["answered"] = True
        print("[Step] Answer sent to user")
        
    elif action == "reject_request":
        reason = state.get("rejection_reason", "Unauthorized query")
        print(f"\nRequest Rejected – {reason}")
        state["answered"] = True
        print("[Step] Request rejected, process finished")

    return state

In [15]:
def run_decision_agent(df, question):
    state = state_template.copy()
    state["request_received"] = True
    print(f"\n=== New Question Received: '{question}' ===")
    for step in range(10):
        print(f"\n\n[Agent Loop] Step {step+1}")
        action = decide_action(state, question)
        if state["authorized"] is False and action == "run_analysis":
            action = "reject_request"
        state = execute_action(action, state, df, question)
        if action == "finish":
            print(f"\n\n[Agent Loop] Finished processing '{question}'")
            break

In [16]:
df = pd.read_csv("/content/drive/MyDrive/hf_models/agent_project/sales_dataset.csv")
df["date"] = pd.to_datetime(df["date"])

test_questions = [
    "What is the total revenue?",
    "What is the average revenue per region?",
    "Revenue by product_category (top 3 categories)?",
    "what is the monthly revenue trend?",
    "Revenue per region per product_category?",
    
    "Show all sales rows.",
    "List every transaction.",
    "Export the table to CSV.",
    "Download entire dataset.",
    "Show all records for North region.",
]

for q in test_questions:
    print("\nQuestion:", q)
    run_decision_agent(df, q)
    print("\n", "-"*60, "\n", "-"*60)


Question: What is the total revenue?

=== New Question Received: 'What is the total revenue?' ===


[Agent Loop] Step 1

[Step] Current action: classify_request
[Step] Classification done – authorized=True


[Agent Loop] Step 2

[Step] Current action: run_analysis
[Step] Asking LLM to generate pandas code...

[LLM Raw Output]
result = df['revenue'].sum()

[Cleaned Code]
result = df['revenue'].sum()
[Step] Analysis done – result ready


[Agent Loop] Step 3

[Step] Current action: answer_user

Answer: Answer to 'What is the total revenue?': 34514
[Step] Answer sent to user


[Agent Loop] Step 4

[Step] Current action: finish


[Agent Loop] Finished processing 'What is the total revenue?'

 ------------------------------------------------------------ 
 ------------------------------------------------------------

Question: What is the average revenue per region?

=== New Question Received: 'What is the average revenue per region?' ===


[Agent Loop] Step 1

[Step] Current action: classif

<string>:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.



[LLM Raw Output]
result = df.groupby(['region', 'product_category'])['revenue'].sum().reset_index()

[Cleaned Code]
result = df.groupby(['region', 'product_category'])['revenue'].sum().reset_index()
[Step] Analysis done – result ready


[Agent Loop] Step 3

[Step] Current action: answer_user

Answer: Answer to 'Revenue per region per product_category?': [{'region': 'Central', 'product_category': 'Books', 'revenue': 77}, {'region': 'Central', 'product_category': 'Clothing', 'revenue': 727}, {'region': 'Central', 'product_category': 'Electronics', 'revenue': 2528}, {'region': 'Central', 'product_category': 'Food', 'revenue': 46}, {'region': 'Central', 'product_category': 'Furniture', 'revenue': 1902}, {'region': 'Central', 'product_category': 'Sports', 'revenue': 470}, {'region': 'East', 'product_category': 'Books', 'revenue': 251}, {'region': 'East', 'product_category': 'Clothing', 'revenue': 531}, {'region': 'East', 'product_category': 'Furniture', 'revenue': 3203}, {'region': 'East',